In [1]:
import pandas as pd
import pickle


In [2]:
frac = 0.02
yr = '2020'

In [3]:
### Cargar modelos

filename = './../fitted_RF/clf1_'+str(frac)+'_'+yr+'_ARG.sav'
clf1 = pickle.load(open(filename, 'rb'))

filename = './../fitted_RF/clf2_'+str(frac)+'_'+yr+'_ARG.sav'
clf2 = pickle.load(open(filename, 'rb'))

filename = './../fitted_RF/clf3_'+str(frac)+'_'+yr+'_ARG.sav'
clf3 = pickle.load(open(filename, 'rb'))


In [4]:
sample_censo_file = './../data/censo_samples/sample_censo_table_f'+str(frac)+'_'+yr+'_ARG.csv'

# Debe contener las siguientes columnas:
# Referidas a las personas:
# [ 'P02', 'P03', 'P05', 'P07', 'P08', 'P09', 'P10', 'PROP', 'CONDACT']

# Referidas a las viviendas y hogares:
# ['V01', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'IX_TOT']

# Referidas al lugar
# ['RADIO_REF_ID']
# El RADIO se usa para determinar Aglomerado y Region del pais.

In [5]:
### CARGAR DATOS DE CENSO
X_censo = pd.read_csv(sample_censo_file).fillna(0)
len(X_censo)/frac


### COMPLETAR INFORMACION (REGIONES/AGLOMERADOS)
dpto_region = pd.read_csv('./../data/info/DPTO_PROV_Region.csv')
AGLO_rk = pd.read_csv('./../data/info/AGLO_rk')
Reg_rk = pd.read_csv('./../data/info/Reg_rk')

X_censo = X_censo.merge(dpto_region[['DPTO', 'Region']]).merge(AGLO_rk[['AGLOMERADO', 'AGLO_rk']]).merge(Reg_rk[['Region', 'Reg_rk']])


In [6]:
### Parte 1

x_cols1 = ['IX_TOT', 'P02', 'P03', 'AGLO_rk', 'Reg_rk', 'V01', 'H05', 'H06',
       'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H16', 'H15', 'PROP', 'H14',
       'H13', 'P07', 'P08', 'P09', 'P10', 'P05', 'CONDACT']
predecir1 = ['CAT_OCUP', 'CAT_INAC', 'CH07']


# Predict
y_out1 = clf1.predict(X_censo[x_cols1].values)
y_censo_fit1 = pd.DataFrame(y_out1, index = X_censo.index, columns=predecir1)
Xy1_censo = pd.concat([X_censo, y_censo_fit1], axis = 1)


In [7]:
### Parte 2

x_cols2 = x_cols1 + predecir1
predecir2 = ['INGRESO', 'INGRESO_NLB', 'INGRESO_JUB', 'INGRESO_SBS']


y_out2 = clf2.predict(Xy1_censo[x_cols2].values)
y_censo_fit2 = pd.DataFrame(y_out2, index = Xy1_censo.index, columns=predecir2)

Xy2_censo = pd.concat([Xy1_censo, y_censo_fit2], axis = 1)

In [8]:
### Parte 3

x_cols3 = x_cols2 + predecir2
predecir3 = ['PP07G1','PP07G_59', 'PP07I', 'PP07J', 'PP07K']


y_out3 = clf3.predict(Xy2_censo[x_cols3].values)
y_censo_fit3 = pd.DataFrame(y_out3, index = Xy2_censo.index, columns=predecir3)

Xy3_censo = pd.concat([Xy2_censo, y_censo_fit3], axis = 1)

In [9]:
### Parte 4

x_cols4 = x_cols3 + predecir3

# Columnas de ingresos. Necesitan una regresion...
columnas_pesos = [u'P21', u'P47T', u'PP08D1', u'TOT_P12', u'T_VI', u'V12_M', u'V2_M', u'V3_M', u'V5_M']
predecir4 = columnas_pesos


In [10]:
# Xy3_censo = pd.read_csv('./../data/yr_samples/RFC3_'+str(frac)+'_'+yr+'_ARG.csv')
import os

qs = ['2020-03-31', '2020-06-30']
for q in qs:

    ## Cargar Modelo
    filename = './../fitted_RF/clf4_'+str(frac)+'_'+q+'_ARG.sav'
    clf4 = pickle.load(open(filename, 'rb'))

    y_out4 = clf4.predict(Xy3_censo[x_cols4].values)
    y_censo_fit4 = pd.DataFrame(y_out4, index = Xy3_censo.index, columns=predecir4)


    Xy4_censo = pd.concat([Xy3_censo, y_censo_fit4], axis = 1)
    
    if not os.path.exists('./../data/results/'):
        os.makedirs('./../data/results/')
    Xy4_censo.to_csv('./../data/results/RFReg_'+str(frac)+'ARG'+str(q)+'.csv', index = False)


In [11]:
Xy4_censo

,VIVIENDA_REF_ID,RADIO_REF_ID,TIPVV,V01,DPTO,HOGAR_REF_ID,H05,H06,H07,H08,...,PP07K,P21,P47T,PP08D1,TOT_P12,T_VI,V12_M,V2_M,V3_M,V5_M
0,8975437,32439,1,1.0,30015,7821450,1,4.0,1,1,...,1.0,4.133762,4.133762,4.133762,0.000000,0.0,0.0,0.0,0.0,0.0
1,8975437,32439,1,1.0,30015,7821450,1,4.0,1,1,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
2,8975437,32439,1,1.0,30015,7821450,1,4.0,1,1,...,1.0,4.454082,4.454082,4.454082,0.000000,0.0,0.0,0.0,0.0,0.0
3,8975437,32439,1,1.0,30015,7821450,1,4.0,1,1,...,4.0,3.870521,3.870521,3.870521,0.000000,0.0,0.0,0.0,0.0,0.0
4,8975437,32439,1,1.0,30015,7821450,1,4.0,1,1,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
892056,1361261,3378,1,1.0,2015,1098147,1,2.0,1,1,...,4.0,0.000000,4.636418,0.000000,4.636418,0.0,0.0,0.0,0.0,0.0
892057,1361296,3378,1,1.0,2015,1098180,1,2.0,1,1,...,1.0,4.636418,4.636418,4.636418,0.000000,0.0,0.0,0.0,0.0,0.0
892058,1361296,3378,1,1.0,2015,1098180,1,2.0,1,1,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
892059,1361296,3378,1,1.0,2015,1098180,1,2.0,1,1,...,1.0,4.937443,4.937443,4.937443,0.000000,0.0,0.0,0.0,0.0,0.0
